In [882]:
import torch
import plotly.graph_objects as go
import os
from plotly.subplots import make_subplots
from typing import Literal
import plotly.colors as pc
from pathlib import Path
import math
import pandas as pd
from IPython.display import HTML
import numpy as np


In [883]:
HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Montserrat:wght@300;400;500;600;700&display=swap');
</style>
""")

# Constants

In [884]:
langs = [
    "fra_Latn",
    "eng_Latn",
    "por_Latn",
    "spa_Latn",
    "jpn_Jpan",
    "zho_Hans",
    "swh_Latn",
    "wol_Latn",
    "hin_Deva",
    "arb_Arab",
    "rus_Cyrl",
]

In [885]:
models = [
    "gemma-3-4b-pt",
    "gemma-3-1b-pt",
    "gemma-3-270m",
]

In [886]:
lang_names = {
    "fra_Latn": "French",
    "eng_Latn": "English",
    "por_Latn": "Portuguese",
    "spa_Latn": "Spanish",
    "jpn_Jpan": "Japanese",
    "zho_Hans": "Chinese",
    "swh_Latn": "Swahili",
    "wol_Latn": "Wolof",
    "hin_Deva": "Hindi",
    "arb_Arab": "Arabic",
    "rus_Cyrl": "Russian",
}

# Style

In [887]:
models_colors = {
    "gemma-3-12b-pt": "#1B3764",
    "gemma-3-4b-pt": "#244A86",
    "gemma-3-1b-pt": "#376FCA",
    "gemma-3-270m": "#4287F5",
}

In [888]:
lang_colors = {
    "fra_Latn": "#8dd3c7",
    "eng_Latn": "#ffffb3",
    "por_Latn": "#bebada",
    "spa_Latn": "#fb8072",
    "jpn_Jpan": "#80b1d3",
    "zho_Hans": "#fdb462",
    "swh_Latn": "#b3de69",
    "wol_Latn": "#fccde5",
    "hin_Deva": "#d9d9d9",
    "arb_Arab": "#ccebc5",
    "rus_Cyrl": "#ffed6f",
}

In [889]:
def style_fig(fig: go.Figure) -> go.Figure:
    font = "Montserrat"

    fig.update_layout(
        template="plotly_white",
        font=dict(family=font, size=14),
        legend=dict(
            x=0.98,
            y=0.02,
            xanchor="right",
            yanchor="bottom",
            bgcolor="rgba(245,245,245,0.4)",
            bordercolor="rgba(0,0,0,0)",  # no border
            font=dict(size=8),
        ),
    )

    fig.update_xaxes(
        title_font=dict(size=16, family=font), tickfont=dict(size=14, family=font)
    )
    fig.update_yaxes(
        title_font=dict(size=16, family=font), tickfont=dict(size=14, family=font)
    )

    return fig

# Utils

In [890]:
def load_logprobs_diff(model, source, target, type: Literal["lang", "trad"]):
    filename = (
        f"../results/translation_task/logprobs_diff_{type}/{model}:{source}:{target}.pt"
    )
    if os.path.exists(filename):
        return torch.load(filename, map_location=torch.device("cpu"))
    else:
        raise FileNotFoundError(
            f"No logprobs_diff file found for {model} with source {source} and target {target}."
        )


def load_all_logprobs_diff(model: str, langs: list[str], type: Literal["lang", "trad"]):
    logprobs_diff_dict = {}
    for source in langs:
        for target in langs:
            if source != target:
                try:
                    logprobs_diff = load_logprobs_diff(model, source, target, type)
                    logprobs_diff_dict[(source, target)] = logprobs_diff
                except FileNotFoundError:
                    continue

    return logprobs_diff_dict

In [891]:
def plot_logprobs_diff(
    logprobs_diffs: dict[tuple[str, str], torch.Tensor],
    langs: list[tuple[str, str]],
    title: str,
):
    cols = 3
    rows = math.ceil(len(langs) / cols) + 1
    colorscale = pc.diverging.BrBG

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[
            "Mean",
            "Mean(English as Source)",
            "Mean(English as Target)",
        ]
        + [
            f"{source.split('_')[0]} -> {target.split('_')[0]}"
            for source, target in langs
        ],
        horizontal_spacing=0.05,
        vertical_spacing=0.05,
    )

    mean_logprobs_diff_all = torch.stack(
        [logprobs_diffs[(source, target)].mean(dim=-1) for source, target in langs],
        dim=0,
    ).mean(dim=0)

    max_logprobs_diff_all = (
        torch.stack(
            [logprobs_diffs[(source, target)].mean(dim=-1) for source, target in langs],
            dim=0,
        )
        .max()
        .item()
    )

    mean_logprobs_diff_source = torch.stack(
        [
            logprobs_diffs[(source, target)].mean(dim=-1)
            for source, target in langs
            if source == "eng_Latn"
        ],
        dim=0,
    ).mean(dim=0)

    mean_logprobs_diff_target = torch.stack(
        [
            logprobs_diffs[(source, target)].mean(dim=-1)
            for source, target in langs
            if target == "eng_Latn"
        ],
        dim=0,
    ).mean(dim=0)

    fig.add_trace(
        go.Heatmap(
            z=mean_logprobs_diff_all.numpy(),
            x=[f"H{i}" for i in range(mean_logprobs_diff_all.shape[1])],
            y=[f"L{i}" for i in range(mean_logprobs_diff_all.shape[0])],
            colorscale=colorscale,
            zmid=0.0,
            zmax=max_logprobs_diff_all,
            zmin=-max_logprobs_diff_all,
            showscale=False,
            xgap=3,
            ygap=1,
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Heatmap(
            z=mean_logprobs_diff_source.numpy(),
            x=[f"H{i}" for i in range(mean_logprobs_diff_source.shape[1])],
            y=[f"L{i}" for i in range(mean_logprobs_diff_source.shape[0])],
            colorscale=colorscale,
            zmid=0.0,
            zmax=max_logprobs_diff_all,
            zmin=-max_logprobs_diff_all,
            showscale=False,
            xgap=3,
            ygap=1,
        ),
        row=1,
        col=2,
    )

    fig.add_trace(
        go.Heatmap(
            z=mean_logprobs_diff_target.numpy(),
            x=[f"H{i}" for i in range(mean_logprobs_diff_target.shape[1])],
            y=[f"L{i}" for i in range(mean_logprobs_diff_target.shape[0])],
            colorscale=colorscale,
            zmid=0.0,
            zmax=max_logprobs_diff_all,
            zmin=-max_logprobs_diff_all,
            showscale=False,
            xgap=4,
            ygap=1,
        ),
        row=1,
        col=3,
    )

    for i, (source, target) in enumerate(langs):
        mean_logprobs_diff = logprobs_diffs[(source, target)].mean(dim=-1)

        fig.add_trace(
            go.Heatmap(
                z=mean_logprobs_diff.numpy(),
                x=[f"H{i}" for i in range(mean_logprobs_diff.shape[1])],
                y=[f"L{i}" for i in range(mean_logprobs_diff.shape[0])],
                colorscale=colorscale,
                zmid=0.0,
                zmax=max_logprobs_diff_all,
                zmin=-max_logprobs_diff_all,
                showscale=False,
                xgap=3,
                ygap=1,
            ),
            row=(i // cols) + 2,
            col=(i % cols) + 1,
        )

    for r in range(1, rows + 1):
        for c in range(1, cols + 1):
            idx = (r - 1) * cols + c

            fig.update_yaxes(
                scaleanchor=f"x{idx if idx > 1 else ''}",
                scaleratio=0.33,
                showgrid=False,
                row=r,
                col=c,
            )
            fig.update_xaxes(showgrid=False, row=r, col=c)

    fig.update_layout(height=400 * rows, width=1000, title_text=title)
    fig = style_fig(fig)

    fig.show()

In [892]:
def plot_logprobs_diff_contribution_source_target(
    logprobs_diffs: dict[tuple[str, str], torch.Tensor],
    langs: list[tuple[str, str]],
    title: str,
):
    contributions = {}
    for source, target in langs:
        logprobs_diff, _ = torch.sort(
            logprobs_diffs[(source, target)].mean(dim=-1).flatten(),
            descending=True,
        )
        contributions[(source, target)] = torch.cumsum(logprobs_diff, dim=0)

    # Make one line plot with all contributions
    fig = go.Figure()

    # Source vs Target contribution

    source_contribution = torch.stack(
        [
            contributions[(source, target)]
            for source, target in langs
            if source == "eng_Latn"
        ],
        dim=0,
    )
    source_mean = source_contribution.mean(dim=0)
    source_std = source_contribution.std(dim=0)

    target_contribution = torch.stack(
        [
            contributions[(source, target)]
            for source, target in langs
            if target == "eng_Latn"
        ],
        dim=0,
    )
    target_mean = target_contribution.mean(dim=0)
    target_std = target_contribution.std(dim=0)

    x = np.arange(1, source_contribution.shape[1] + 1)

    fig.add_trace(
        go.Scatter(
            x=x,
            y=source_mean.numpy(),
            mode="lines",
            name="Source ",
            line=dict(width=2, color="#1b998b"),
        ),
    )

    fig.add_trace(
        go.Scatter(
            x=np.concatenate(
                [x, x[::-1]],
            ),
            y=np.concatenate(
                [
                    (source_mean + source_std).numpy(),
                    (source_mean - source_std).numpy()[::-1],
                ],
            ),
            fill="toself",
            hoverinfo="skip",
            mode="lines",
            line=dict(width=0, color="#1b998b"),
            showlegend=False,
        ),
    )

    fig.add_trace(
        go.Scatter(
            x=x,
            y=target_mean.numpy(),
            mode="lines",
            name="Target ",
            line=dict(width=2, color="#f46036"),
        ),
    )

    fig.add_trace(
        go.Scatter(
            x=np.concatenate(
                [x, x[::-1]],
            ),
            y=np.concatenate(
                [
                    (target_mean + target_std).numpy(),
                    (target_mean - target_std).numpy()[::-1],
                ],
            ),
            fill="toself",
            mode="lines",
            hoverinfo="skip",
            line=dict(width=0, color="#f46036"),
            showlegend=False,
        ),
    )

    fig.update_layout(
        title_text=title,
        xaxis_title="Number of Heads",
        yaxis_title="Cumulative Logprob Difference",
    )

    fig = style_fig(fig)

    fig.show()

In [893]:
def plot_logprobs_diff_contribution_lang(
    logprobs_diffs: dict[tuple[str, str], torch.Tensor],
    langs: list[tuple[str, str]],
    title: str,
):
    contributions = {}
    for source, target in langs:
        mean_flatened = logprobs_diffs[(source, target)].mean(dim=-1).flatten()
        # softmaxed = torch.softmax(mean_flatened, dim=0)
        sorted = torch.sort(mean_flatened, descending=True).values

        # contributions[(source, target)] = torch.cumsum(logprobs_diff, dim=1)
        contributions[(source, target)] = torch.cumsum(sorted, dim=0)

    # Lang pair contributions

    lang_set = set()
    for source, target in langs:
        lang_set.add(source)
        lang_set.add(target)

    lang_set.remove("eng_Latn")

    fig = make_subplots(
        rows=1, cols=2, subplot_titles=["English as Source", "English as Target"]
    )

    x = np.arange(1, contributions[next(iter(contributions))].shape[0] + 1)

    for i, lang in enumerate(lang_set):
        mean_source = contributions[("eng_Latn", lang)]
        mean_target = contributions[(lang, "eng_Latn")]

        fig.add_trace(
            go.Scatter(
                x=x,
                y=mean_source.numpy(),
                mode="lines",
                name=lang.split("_")[0].title(),
                line=dict(width=2, color=lang_colors.get(lang, "black")),
                showlegend=True,
                legendgroup=str(i // 3),
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=x,
                y=mean_target.numpy(),
                mode="lines",
                name=lang.split("_")[0].title(),
                line=dict(width=2, color=lang_colors.get(lang, "black")),
                showlegend=False,
            ),
            row=1,
            col=2,
        )

    # Layout updates

    fig.update_xaxes(title_text="Number of Heads")
    fig.update_layout(
        title_text=title,
        yaxis_title="Cumulative Logprob Difference",
        legend=dict(orientation="h"),
    )

    fig = style_fig(fig)

    fig.show()

In [894]:
def plot_scaling_diff(
    logprobs_diffs: dict[str, dict[tuple[str, str], torch.Tensor]],
    title: str,
    get_model_title: callable,
):
    fig = go.Figure()

    for model_name, logprobs_diff in logprobs_diffs.items():
        mean_logprobs_diff_all = (
            torch.stack(
                [
                    logprobs_diff[(source, target)].mean(dim=-1)
                    for source, target in logprobs_diff.keys()
                ],
                dim=0,
            ).mean(dim=0)
        ).clamp(min=0)

        # mean_logprobs_diff_all = (
        #     torch.stack(
        #         [
        #             logprobs_diff[(source, target)].mean(dim=-1)
        #             for source, target in logprobs_diff.keys()
        #         ],
        #         dim=0,
        #     ).mean(dim=0)
        # ).softmax(dim=0)

        cumulative_logprobs_diff_all = (
            torch.cumsum(
                torch.sort(
                    mean_logprobs_diff_all.flatten(),
                    descending=True,
                ).values,
                dim=0,
            )
            / mean_logprobs_diff_all.sum()
        )

        # Add a 0 at the beginning for 0 heads
        cumulative_logprobs_diff_all = cumulative_logprobs_diff_all.numpy()
        x = np.arange(1, len(cumulative_logprobs_diff_all) + 1)

        fig.add_trace(
            go.Scatter(
                x=x,
                y=cumulative_logprobs_diff_all[:20],
                mode="lines",
                name=get_model_title(model_name),
                line=dict(width=2, color=models_colors.get(model_name, "black")),
            ),
        )

        # Add dashed line for y = a * sqrt(x)

    fig.update_layout(
        title_text=title,
        xaxis_title="Number of Heads",
        yaxis_title="Mean Logprob Difference",
    )

    fig = style_fig(fig)

    fig.show()

In [895]:
def load_scores(
    model: str, score_type: Literal["bleu", "chrf"]
) -> tuple[
    dict[tuple[str, str], float],
    dict[tuple[str, str], dict[tuple[int, int], float]],
]:
    generation_path = Path("../results/translation_task/generations/")
    baseline_score, intervention_scores = {}, {}

    for source in langs:
        for target in langs:
            for lang_head in range(6):
                for trad_head in range(6):
                    filename = (
                        generation_path
                        / f"{model}:{source}:{target}:{source}:{target}:{lang_head}:{trad_head}.csv"
                    )
                    if filename.exists():
                        if (source, target) not in intervention_scores:
                            intervention_scores[(source, target)] = {}

                        df = pd.read_csv(filename)

                        baseline_score[(source, target)] = df[
                            f"{score_type}_baseline"
                        ].mean()
                        intervention_scores[(source, target)][
                            (lang_head, trad_head)
                        ] = df[f"{score_type}_function_vector"].mean()

In [896]:
def plot_scores(
    scores: dict[tuple[str, str], dict[tuple[int, int], float]], title: str
):
    cols = 3
    rows = math.ceil(len(scores) / cols) + 1

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=["", "Overall Mean", ""]
        + [
            f"{source.split('_')[0]} -> {target.split('_')[0]}"
            for source, target in scores.keys()
        ],
    )

# Langs

## Gemma-3

### Scaling

In [897]:
logprobs_diff_lang = {
    "gemma-3-4b-pt": load_all_logprobs_diff("gemma-3-4b-pt", langs, type="lang"),
    "gemma-3-1b-pt": load_all_logprobs_diff("gemma-3-1b-pt", langs, type="lang"),
    "gemma-3-270m": load_all_logprobs_diff("gemma-3-270m", langs, type="lang"),
}

In [898]:
plot_scaling_diff(
    logprobs_diff_lang,
    title="Concentration of Logprob Differences for Language vs. Translation Heads",
    get_model_title=lambda model_name: model_name.split("-")[2].upper(),
)

### Gemma-3-12b-pt

In [899]:
logprobs_diff_lang = load_all_logprobs_diff("gemma-3-12b-pt", langs, type="lang")

In [900]:
plot_logprobs_diff(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Traduction gemma-3-12b-pt",
)

In [901]:
plot_logprobs_diff_contribution_source_target(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Contribution Language gemma-3-12b-pt",
)

In [902]:
plot_logprobs_diff_contribution_lang(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Contribution Language gemma-3-12b-pt",
)

### Gemma-3-4b-pt

In [903]:
logprobs_diff_lang = load_all_logprobs_diff("gemma-3-4b-pt", langs, type="lang")

In [904]:
plot_logprobs_diff(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Traduction gemma-3-4b-pt",
)

In [905]:
plot_logprobs_diff_contribution_source_target(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Contribution Language gemma-3-4b-pt",
)

In [906]:
plot_logprobs_diff_contribution_lang(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Contribution Language gemma-3-4b-pt",
)

### Gemma-3-1b-pt

In [907]:
logprobs_diff_lang = load_all_logprobs_diff("gemma-3-1b-pt", langs, type="lang")

In [908]:
plot_logprobs_diff(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Language gemma-3-1b-pt",
)

In [909]:
plot_logprobs_diff_contribution_source_target(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Contribution Language gemma-3-1b-pt",
)

In [910]:
plot_logprobs_diff_contribution_lang(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Contribution Language gemma-3-1b-pt",
)

### Gemma-3-270m-pt

In [911]:
logprobs_diff_lang = load_all_logprobs_diff("gemma-3-270m", langs, type="lang")

In [912]:
plot_logprobs_diff(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Language gemma-3-270m-pt",
)

In [913]:
plot_logprobs_diff_contribution_source_target(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Contribution Language gemma-3-270m-pt",
)

In [914]:
plot_logprobs_diff_contribution_lang(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Contribution Language gemma-3-270m",
)

# Traduction

## Gemma-3

### Scaling

In [915]:
logprobs_diff_trad = {
    "gemma-3-4b-pt": load_all_logprobs_diff("gemma-3-4b-pt", langs, type="trad"),
    "gemma-3-1b-pt": load_all_logprobs_diff("gemma-3-1b-pt", langs, type="trad"),
    "gemma-3-270m": load_all_logprobs_diff("gemma-3-270m", langs, type="trad"),
}

In [916]:
plot_scaling_diff(
    logprobs_diff_trad,
    title="Concentration of Logprob Differences for Translation vs. Language Heads",
    get_model_title=lambda model_name: model_name.split("-")[2].upper(),
)

### Gemma-3-12b-pt

In [917]:
logprobs_diff_trad = load_all_logprobs_diff("gemma-3-12b-pt", langs, type="trad")

In [918]:
plot_logprobs_diff(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    title="Logprobs Diff Traduction gemma-3-12b-pt",
)

### Gemma-3-4b-pt

In [919]:
logprobs_diff_trad = load_all_logprobs_diff("gemma-3-4b-pt", langs, type="trad")

In [920]:
plot_logprobs_diff(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    title="Logprobs Diff Traduction gemma-3-4b-pt",
)

In [921]:
plot_logprobs_diff_contribution_source_target(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    title="Logprobs Diff Contribution Traduction gemma-3-4b-pt",
)

In [922]:
plot_logprobs_diff_contribution_lang(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    title="Logprobs Diff Contribution Traduction gemma-3-4b-pt",
)

### Gemma-3-1b-pt

In [923]:
logprobs_diff_trad = load_all_logprobs_diff("gemma-3-1b-pt", langs, type="trad")

In [924]:
plot_logprobs_diff(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    title="Logprobs Diff Traduction gemma-3-1b-pt",
)

In [925]:
plot_logprobs_diff_contribution_source_target(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    title="Logprobs Diff Contribution Traduction gemma-3-1b-pt",
)

In [926]:
plot_logprobs_diff_contribution_lang(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    title="Logprobs Diff Contribution Traduction gemma-3-1b-pt",
)

### Gemma-3-270m-pt

In [927]:
logprobs_diff_trad = load_all_logprobs_diff("gemma-3-270m", langs, type="trad")

In [928]:
plot_logprobs_diff(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    title="Logprobs Diff Traduction gemma-3-270m-pt",
)

In [929]:
plot_logprobs_diff_contribution_source_target(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    title="Logprobs Diff Contribution Traduction gemma-3-270m-pt",
)

In [930]:
plot_logprobs_diff_contribution_lang(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    title="Logprobs Diff Contribution Traduction gemma-3-270m",
)